# 02 Jan Milestone — Machine Learning (Regression)

This notebook completes the **02 Jan milestone** in the project timeline: applying ML methods to the dataset.

It trains multiple regression models to predict **Food_Price_Index** using economic indicators (e.g., **USD_TRY**) and time/lag features.

Works with the current project dataset (`data/food_inflation_data.csv`). If you replace the CSV with official sources, keep the same column names or update the notebook accordingly.


In [ ]:
# If running locally, install deps once:
# !pip install -r ../requirements.txt

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

RANDOM_STATE = 42
SAVE_FIGS = True
IMG_DIR = "../images"
os.makedirs(IMG_DIR, exist_ok=True)

DATA_PATH = "../data/food_inflation_data.csv"
df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COLS = ["Bread_Price", "Milk_Price", "Meat_Price"]
INDEX_COLS = ["Bread_Index", "Milk_Index", "Meat_Index"]
if "Food_Price_Index" not in df.columns:
    if all(col in df.columns for col in INDEX_COLS):
        df["Food_Price_Index"] = df[INDEX_COLS].mean(axis=1)
    elif all(col in df.columns for col in PRICE_COLS):
        base_row = df.iloc[0]
        for col in PRICE_COLS:
            df[f"{col}_Index"] = df[col] / base_row[col] * 100
        df["Food_Price_Index"] = df[[f"{col}_Index" for col in PRICE_COLS]].mean(axis=1)
    else:
        raise ValueError("Expected price or index columns for food items.")

df.head(), df.shape


## 1) Basic cleaning + time features
We treat this as a **time series regression** problem, so we keep ordering by date and avoid random splits.

In [ ]:
# --- Date parsing ---
# Accept common variants: Date, date, Month, timestamp
date_col_candidates = [c for c in df.columns if c.lower() in {"date","month","timestamp","time"}]
if not date_col_candidates:
    raise ValueError(f"No date column found. Available columns: {list(df.columns)}")
date_col = date_col_candidates[0]

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True)

# --- Target column ---
target_candidates = [c for c in df.columns if c.lower() in {"food_price_index","foodpriceindex","food_index","food_index_value"}]
if not target_candidates:
    raise ValueError("Target column not found. Expected something like 'Food_Price_Index'.")
target_col = target_candidates[0]

# --- Time features ---
df["year"] = df[date_col].dt.year
df["month"] = df[date_col].dt.month
df["t"] = np.arange(len(df))  # monotonic time index

df[[date_col, target_col, "year", "month", "t"]].head()

## 2) Feature engineering (lags + optional macro features)
We add **lag features** to capture inertia/seasonality (1-month and 12-month). If real data later includes extra macro variables (e.g., Inflation_Rate), they will be used automatically.

In [ ]:
# Candidate numeric features (exclude target + date)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

# Add lag features on target (safe for monthly data)
df[f"{target_col}_lag_1"] = df[target_col].shift(1)
df[f"{target_col}_lag_12"] = df[target_col].shift(12)

# Rolling means (optional but helps smoother predictions)
df[f"{target_col}_roll_3"] = df[target_col].shift(1).rolling(3).mean()
df[f"{target_col}_roll_6"] = df[target_col].shift(1).rolling(6).mean()

# Refresh numeric feature list (include lags/rolls + existing macro variables like USD_TRY)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

# Categorical features (if later you add things like City / Category)
cat_cols = [c for c in df.columns if df[c].dtype == "object" and c not in {date_col, target_col}]

numeric_cols, cat_cols


## 3) Train/Validation/Test split (time-respecting)
We use a simple **80/20 holdout** split at the end of the series.

- Train: first 80%
- Test: last 20%

This avoids leakage and matches time series evaluation best practices.

In [ ]:
# Drop rows created by lagging at the beginning
df_model = df.dropna(subset=[f"{target_col}_lag_1"]).copy()  # keep lag_1 at minimum
df_model = df_model.reset_index(drop=True)

X = df_model[numeric_cols + cat_cols]
y = df_model[target_col]
dates = df_model[date_col]

split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = dates.iloc[split_idx:]

len(X_train), len(X_test), dates.iloc[0], dates.iloc[-1]

## 4) Baselines
A good ML section always compares against a baseline.

**Baseline 1: Naive last value** (predict next month equals previous month).

**Baseline 2: Seasonal naive** (predict equals value from 12 months ago, if available).

In [ ]:
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2}

# Baseline 1: last value (lag_1)
y_pred_naive = X_test[f"{target_col}_lag_1"].values
baseline_naive = metrics(y_test, y_pred_naive)

# Baseline 2: seasonal naive (lag_12) if exists and has no NaN in test
if f"{target_col}_lag_12" in X_test.columns and not X_test[f"{target_col}_lag_12"].isna().any():
    y_pred_seasonal = X_test[f"{target_col}_lag_12"].values
    baseline_seasonal = metrics(y_test, y_pred_seasonal)
else:
    baseline_seasonal = None

baseline_naive, baseline_seasonal

## 5) ML models (regression)
We train a few models (simple → stronger):

- Linear Regression
- Ridge / Lasso (regularized linear)
- Random Forest (non-linear)
- HistGradientBoosting (strong for tabular)

All are trained with a **sklearn Pipeline** that handles missing values and encoding.

In [ ]:
# Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=1.0)": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso(alpha=0.001)": Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1, max_depth=None
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    res = metrics(y_test, y_pred)
    res["Model"] = name
    results.append(res)
    fitted[name] = (pipe, y_pred)

results_df = pd.DataFrame(results).set_index("Model").sort_values("RMSE")
results_df

## 6) Plot: Actual vs Predicted (best model)
We plot the best model (lowest RMSE) against actual values for the test period.

In [ ]:
best_model_name = results_df.index[0]
best_pipe, best_pred = fitted[best_model_name]

plt.figure(figsize=(10,4))
plt.plot(dates_test, y_test.values, label="Actual")
plt.plot(dates_test, best_pred, label=f"Predicted ({best_model_name})")
plt.title("Food Price Index — Actual vs Predicted (Test)")
plt.xlabel("Date")
plt.ylabel(target_col)
plt.legend()
plt.tight_layout()

if SAVE_FIGS:
    outpath = os.path.join(IMG_DIR, "ml_actual_vs_pred.png")
    plt.savefig(outpath, dpi=200)

plt.show()

best_model_name

## 7) Residual analysis
Residuals help show where the model systematically over/under-predicts.

In [ ]:
residuals = y_test.values - best_pred

plt.figure(figsize=(10,4))
plt.plot(dates_test, residuals)
plt.axhline(0, linestyle="--")
plt.title("Residuals over time (Actual - Predicted)")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.tight_layout()

if SAVE_FIGS:
    outpath = os.path.join(IMG_DIR, "ml_residuals_over_time.png")
    plt.savefig(outpath, dpi=200)

plt.show()

plt.figure(figsize=(6,4))
plt.scatter(best_pred, residuals)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.tight_layout()

if SAVE_FIGS:
    outpath = os.path.join(IMG_DIR, "ml_residuals_vs_pred.png")
    plt.savefig(outpath, dpi=200)

plt.show()

## 8) Feature importance / coefficients
- For **linear models**, we show top coefficients.
- For **tree models**, we show feature importances (approx).

In [ ]:
import numpy as np

model_obj = best_pipe.named_steps["model"]

# Get feature names after preprocessing (sklearn >= 1.0)
feature_names = []
try:
    num_names = numeric_cols
    feature_names.extend(num_names)

    if cat_cols:
        ohe = best_pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
        cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
        feature_names.extend(cat_feature_names)
except Exception as e:
    feature_names = None

def plot_top_features(names, values, title, top_n=15, filename=None):
    idx = np.argsort(np.abs(values))[-top_n:]
    plt.figure(figsize=(8,5))
    plt.barh(np.array(names)[idx], np.array(values)[idx])
    plt.title(title)
    plt.tight_layout()
    if SAVE_FIGS and filename:
        plt.savefig(os.path.join(IMG_DIR, filename), dpi=200)
    plt.show()

if feature_names is not None:
    if hasattr(model_obj, "coef_"):
        plot_top_features(feature_names, model_obj.coef_, f"Top coefficients ({best_model_name})", filename="ml_top_coeffs.png")
    elif hasattr(model_obj, "feature_importances_"):
        plot_top_features(feature_names, model_obj.feature_importances_, f"Top feature importances ({best_model_name})", filename="ml_feature_importance.png")
else:
    print("Could not extract feature names; skipping feature plot.")

## 9) Summary (what to report)
In your report/README, you can briefly include:

- Baseline metrics (naive last-month)
- Best ML model metrics
- A short sentence interpreting results (e.g., USD/TRY and lag features explain much of the variance)
- The plot exported to `images/`

In [ ]:
summary = {
    "Baseline_Naive": baseline_naive,
    "Baseline_Seasonal": baseline_seasonal,
    "Best_Model": best_model_name,
    "Best_Model_Metrics": results_df.loc[best_model_name].to_dict()
}
summary